In [1]:
import sqlite3
import pandas as pd
import os
import csv

In [ ]:
DATABASE = "....../v33_koondkorpus_transaktsioonid_v04_2.db"

LINE_DATA_TABLE = "lines_class_info4"

FILTERED_CLASS_TABLE = "lines_class_info4_n80"

DATA_DIR = "../data/"

LARGE_DATA_FILE = DATA_DIR + "n80_examples_large_v01.csv"



## see teeb korrektse uue andmefaili v2

### andmetabelid

In [3]:
conn = sqlite3.connect(DATABASE)
cursor = conn.cursor()

### graafiku punktide info

In [4]:
query = f"""SELECT verb, verb_compound, morph_case, log2_ratio, level,unique_lemmas, ann_unique_lemmas, 
            not_ann_unique_lemmas, olulisus, my_tag, other_tags, annotated, not_annotated, verb_case_count
            FROM {LINE_DATA_TABLE}
            """

class_info = pd.read_sql(query, conn)
class_info

,verb,verb_compound,morph_case,log2_ratio,level,unique_lemmas,ann_unique_lemmas,not_ann_unique_lemmas,olulisus,my_tag,other_tags,annotated,not_annotated,verb_case_count
0,aasima,,ad,-9.965784,-,1,NaN,5.0,-,0,1,1,7,8
1,abistama,,all,-9.965784,-,1,NaN,12.0,-,0,1,1,15,16
2,aeglustama,,all,-9.965784,-,1,NaN,7.0,-,0,1,1,8,9
3,aerutama,,in,-9.965784,-,1,NaN,5.0,-,0,3,3,10,13
4,aevastama,,in,9.965784,-,1,1.0,6.0,-,1,0,1,6,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20935,õnnestuma,,ad,-1.971847,-,1313,331.0,2522.0,-,1022,4009,5031,17243,22274
20936,õppima,,in,5.233872,n90,530,463.0,985.0,0.0,5720,152,5872,5891,11763
20937,ütlema,,ad,-5.370614,n10,415,104.0,1162.0,0.0,295,12205,12500,9966,22466
20938,ütlema,,all,0.271387,n70,1131,189.0,2337.0,1.0,9354,7750,17104,40742,57846


### võtta ainult n80 tsooni lõksud

In [5]:
filtered_class = class_info[class_info["level"]=="n80"]
filtered_class = filtered_class.sort_values(["olulisus"])

In [6]:
filtered_class['olulisus'] = filtered_class['olulisus'].astype(float)

In [7]:
filtered_class

,verb,verb_compound,morph_case,log2_ratio,level,unique_lemmas,ann_unique_lemmas,not_ann_unique_lemmas,olulisus,my_tag,other_tags,annotated,not_annotated,verb_case_count
19889,peatuma,,in,3.551257,n80,282,257.0,337.0,0.00000,973,83,1056,858,1914
19994,möllama,,in,3.921070,n80,196,177.0,261.0,0.00000,409,27,436,495,931
20637,leiduma,,in,3.086569,n80,518,415.0,1689.0,0.00000,1614,190,1804,4760,6564
20673,naasma,,el,3.210249,n80,287,236.0,222.0,0.00000,907,98,1005,439,1444
19859,süttima,,in,3.879146,n80,199,177.0,233.0,0.00000,515,35,550,661,1211
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18469,nappima,,in,3.496426,n80,140,114.0,198.0,0.00006,316,28,344,382,726
18887,sõitma,edasi,el,5.741467,n80,55,53.0,26.0,0.00007,107,2,109,30,139
20015,teatama,,ill,5.235216,n80,54,51.0,82.0,0.00008,113,3,116,640,756
20194,ootama,,ill,3.882643,n80,108,93.0,132.0,0.00008,236,16,252,231,483


In [8]:
filtered_class.to_sql(FILTERED_CLASS_TABLE, conn, if_exists="replace", index=False)

335

### võtta spatial_obl tabelist näitelaused koos vajaliku infoga - pole päris õige päring

In [27]:
# spatial obl_tabelist lõksud, mis on lines_class_info4_n80 tabelis
# iga lõksu kohta max 500 lemmat ja iga unikaalse lemma kohta 1 näide


query = """
SELECT head_id, form, lemma, verb, verb_compound, morph_case, sentence_id, sentence, timex_tag, ekilex_tag, ner_tag
FROM (
    SELECT t2.*,
           ROW_NUMBER() OVER (
               PARTITION BY t2.verb, t2.verb_compound, t2.morph_case, t2.lemma
               ORDER BY t2.lemma
           ) AS rn_lemma,
           ROW_NUMBER() OVER (
               PARTITION BY t2.verb, t2.verb_compound, t2.morph_case
               ORDER BY t2.lemma
           ) AS rn_limit
    FROM spatial_obl t2
    INNER JOIN lines_class_info4_n80 fc
        ON t2.verb = fc.verb
       AND t2.verb_compound = fc.verb_compound
       AND t2.morph_case = fc.morph_case
    WHERE t2.timex_tag IS NULL
) sub
WHERE rn_lemma = 1        -- ensures only one row per lemma
  AND rn_limit <= 500     -- ensures max 500 rows per verb+verb_comp+morph_case
ORDER BY verb, verb_compound, morph_case;


"""


spatial_obl_ex = pd.read_sql(query, conn)


In [1]:
spatial_obl_ex

In [30]:
# shuffle
df = spatial_obl_ex.sample(frac=1)

In [31]:
df.to_csv(LARGE_DATA_FILE, encoding="utf-8", index = False,sep=",", quoting=csv.QUOTE_MINIMAL)

In [33]:
conn.close()

In [2]:
df2 = pd.read_csv(LARGE_DATA_FILE, encoding="utf-8", sep=",")

In [2]:
counts2 = df2.groupby(['verb','verb_compound', 'morph_case'], dropna=False).size().reset_index(name='count').sort_values('count', ascending=False)
#counts2